# Cleansing: เตรียมไฟล์ UNSW-NB15

โค้ดทำความสะอาดอยู่ใน notebook นี้ เพื่อให้เห็นทุกขั้น ไม่เรียก `cleanse_dataset()` จากไฟล์ภายนอก

งานนี้ปรับรูปแบบข้อความ/ชื่อคอลัมน์ แปลง sentinel เป็น missing เพิ่มรหัสแถวและทำเครื่องหมายซ้ำ แล้วบันทึก CSV แบบแบ่ง chunk โดยไม่ลบแถวหรือ clip outlier อัตโนมัติ

> ต้องรัน EDA และตรวจไฟล์ raw ก่อน ไฟล์ต้นฉบับจะไม่ถูกแก้ไข

Notebook นี้ไม่มีกราฟ เพราะผลที่ต้องตรวจเป็นจำนวนแถว/เซลล์และการเปลี่ยนแปลงทีละไฟล์ ตาราง audit เหมาะกับการเห็นยอดก่อน-หลังและตามกลับไปยัง chunk/คอลัมน์ที่เปลี่ยนได้ตรงกว่า

## 1. เตรียม path และตรวจไฟล์ต้นทาง

รองรับ CSV/CSV.GZ/TXT ใน `raw/` ตารางที่แสดงมีชื่อไฟล์ ขนาดเป็น byte และ SHA-256 ซึ่งเป็นลายนิ้วมือของไฟล์ ใช้ยืนยันว่าเลือกไฟล์ต้นทางถูกชุดและไฟล์ไม่ถูกเปลี่ยนระหว่าง cleansing; ถ้า hash ก่อน-หลังไม่ตรง ขั้นตอนจะหยุดแทนการรับรองผลที่อ้างอิง raw คนละรุ่น รอบปัจจุบันพบไฟล์ official Train และ Test ทั้งสองไฟล์

In [1]:
from pathlib import Path
import hashlib
import re
import pandas as pd
from IPython.display import display

current = Path.cwd().resolve()
project_root = next((path for path in (current, *current.parents)
                     if (path / 'data-pipeline' / 'UNSW-NB15' / 'README.md').is_file()), None)
if project_root is None:
    raise FileNotFoundError('เปิด notebook จากภายในโฟลเดอร์โปรเจกต์ CSI_ML_Final')

dataset_dir = project_root / 'data-pipeline' / 'UNSW-NB15'
raw_dir = dataset_dir / 'raw'
output_dir = dataset_dir / 'output' / 'cleansing'
chunk_size = 100_000
supported_suffixes = {'.csv', '.gz', '.txt'}
source_files = sorted(path for path in raw_dir.rglob('*')
                      if path.is_file() and path.suffix.lower() in supported_suffixes)
if not source_files:
    raise FileNotFoundError(f'ไม่พบ CSV ต้นทางใน {raw_dir}; วางไฟล์ UNSW-NB15 ก่อน')

def sha256_file(file_path):
    digest = hashlib.sha256()
    with file_path.open('rb') as file:
        for block in iter(lambda: file.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def make_manifest(paths):
    return pd.DataFrame([
        {'file': path.relative_to(raw_dir).as_posix(),
         'bytes': path.stat().st_size, 'sha256': sha256_file(path)}
        for path in paths
    ], columns=['file', 'bytes', 'sha256'])

manifest_before = make_manifest(source_files)
display(manifest_before[['file', 'bytes', 'sha256']])

,file,bytes,sha256
0,UNSW_NB15_testing-set.csv,15380797,aa807a145f06f6883101bb649ce4344b632cc2828abd76...
1,UNSW_NB15_training-set.csv,32293015,5e3eb6037dab1c7eabd79d3358ab89f76c11003979e21c...


## 2. กติกาปรับข้อความและ missing

ตัด BOM/ช่องว่างรอบชื่อคอลัมน์และข้อความ หากชื่อชนกันหลัง normalize จะหยุดให้ตรวจต้นทาง ข้อความว่างและ sentinel ที่สื่อว่าไม่มีค่าให้เป็น missing ไม่ใช่เลขศูนย์

ผลของ block นี้เป็นข้อความบอกจำนวน sentinel ที่เตรียมกติกาไว้ (11 รูปแบบ) และยกตัวอย่าง เช่นช่องว่างหรือ `n/a`; เป็นจำนวนชนิดข้อความที่ระบบรู้จัก ไม่ใช่จำนวนเซลล์ที่พบในข้อมูลจริง จำนวนที่พบจริงให้ดูใน audit ช่วงท้าย ผลรันครั้งนี้พบ sentinel ที่ถูกแทนเป็น missing 0 ช่อง จึงไม่มีค่าข้อมูลถูกเปลี่ยนเป็น missing และเลข 0 ยังคงเป็นเลข 0

In [2]:
missing_words = {
    '', 'nan', 'null', 'none', 'n/a',
    'inf', '+inf', '-inf', 'infinity', '+infinity', '-infinity',
}

def clean_chunk(raw_chunk):
    cleaned = raw_chunk.copy()
    cleaned.columns = [str(name).replace('\ufeff', '').strip()
                       for name in cleaned.columns]
    if len(set(cleaned.columns)) != len(cleaned.columns):
        raise ValueError('ชื่อคอลัมน์ซ้ำหลังตัด BOM/ช่องว่าง; ตรวจ schema ต้นทาง')

    for column in cleaned.columns:
        text = cleaned[column].astype('string').str.strip()
        is_missing_word = text.str.lower().isin(missing_words)
        cleaned[column] = text.mask(is_missing_word, pd.NA)
    return cleaned

print('จำนวน sentinel ที่กำหนด:', len(missing_words))
print('ตัวอย่าง: ช่องว่างหรือข้อความ n/a จะกลายเป็น missing; ค่า 0 ยังคงเป็น 0')

จำนวน sentinel ที่กำหนด: 11
ตัวอย่าง: ช่องว่างหรือข้อความ n/a จะกลายเป็น missing; ค่า 0 ยังคงเป็น 0


## 3. อ่านทีละ chunk เพิ่มรหัสแถวและทำเครื่องหมาย record ซ้ำ

แต่ละแถวจะได้ `orig_row_id` ที่ประกอบจากชื่อไฟล์และตำแหน่งเดิม ส่วน `_duplicate_flag` บอกว่าเนื้อหาแถวนั้นซ้ำกับแถวก่อนหน้าในไฟล์เดียวกัน เราเก็บแถวซ้ำไว้เพื่อให้ตัดสินใจภายหลังได้จากรายงาน การอ่านข้อความ `เสร็จ: ... (จำนวนแถว)` หลังรันให้ดูว่าอ่านและเขียนแต่ละไฟล์ครบกี่แถว; Train ที่มีมากกว่า 100,000 แถวถูกแบ่งประมวลผลหลาย chunk เพื่อจำกัดหน่วยความจำ รหัสแถวและ duplicate flag เป็น metadata สำหรับตรวจสอบย้อนกลับ ไม่ใช่ feature สำหรับโมเดล

In [3]:
output_dir.mkdir(parents=True, exist_ok=True)
audit_rows = []

for source_path in source_files:
    relative_name = source_path.relative_to(raw_dir).as_posix()
    stem = source_path.name
    for suffix in ('.gz', '.csv', '.txt'):
        if stem.lower().endswith(suffix):
            stem = stem[:-len(suffix)]
    stem = re.sub(r'[^A-Za-z0-9_.-]+', '_', stem)
    output_path = output_dir / f'{stem}.csv'
    temporary_path = output_dir / f'.{stem}.partial.csv'
    temporary_path.unlink(missing_ok=True)

    read_options = {'dtype': 'string', 'chunksize': chunk_size}
    if source_path.suffix.lower() == '.txt':
        read_options.update({'sep': None, 'engine': 'python'})
    else:
        read_options['low_memory'] = False

    rows_written = 0
    missing_cells = 0
    duplicate_rows = 0
    seen_hashes = set()
    wrote_header = False
    try:
        for chunk in pd.read_csv(source_path, **read_options):
            chunk = clean_chunk(chunk)
            chunk_rows = len(chunk)

            # hash ก่อนเพิ่ม metadata เพื่อเทียบเฉพาะเนื้อหาต้นทางที่ normalize แล้ว
            row_hashes = pd.util.hash_pandas_object(chunk, index=False).astype('uint64')
            duplicate_flags = []
            for row_hash in row_hashes:
                hash_value = int(row_hash)
                duplicate_flags.append(hash_value in seen_hashes)
                seen_hashes.add(hash_value)
            duplicate_rows += sum(duplicate_flags)

            chunk.insert(0, 'orig_row_id',
                         [f'{relative_name}:{row_number:09d}'
                          for row_number in range(rows_written, rows_written + chunk_rows)])
            chunk.insert(1, '_duplicate_flag', duplicate_flags)
            missing_cells += int(chunk.isna().sum().sum())

            # เขียนทีละ chunk และเขียน header เฉพาะครั้งแรก
            chunk.to_csv(
                temporary_path,
                mode='a' if wrote_header else 'w',
                header=not wrote_header,
                index=False,
                encoding='utf-8',
            )
            wrote_header = True
            rows_written += chunk_rows

        if not wrote_header:
            raise ValueError(f'ไม่พบ record ใน {source_path.name}')
        temporary_path.replace(output_path)
    except Exception:
        temporary_path.unlink(missing_ok=True)
        raise

    audit_rows.append({
        'file': relative_name, 'rows_in': rows_written, 'rows_out': rows_written,
        'rows_removed': 0, 'duplicate_rows_flagged': duplicate_rows,
        'missing_cells': missing_cells, 'output_file': output_path.name,
    })
    print(f'เสร็จ: {source_path.name} → {output_path.name} ({rows_written:,} แถว)')

เสร็จ: UNSW_NB15_testing-set.csv → UNSW_NB15_testing-set.csv (82,332 แถว)
เสร็จ: UNSW_NB15_training-set.csv → UNSW_NB15_training-set.csv (175,341 แถว)


## 4. ตรวจยอดและบันทึกรายงาน

ตรวจ manifest ซ้ำหลังประมวลผล หาก raw เปลี่ยนระหว่างทางจะไม่ประกาศว่ารอบนี้สำเร็จ ตารางสรุปแสดง `rows_in`, `rows_out`, `rows_removed`, duplicate ที่ทำเครื่องหมาย, missing cells และชื่อไฟล์ output; ข้อความใต้ตารางบอกตำแหน่ง audit/manifest และจำนวนแถวที่ลบจริง ผลรันนี้ Test 82,332 แถว และ Train 175,341 แถว คงจำนวนแถวครบทั้งคู่, ลบ 0 แถว, ทำเครื่องหมาย duplicate 0 แถว และเหลือ missing 0 ช่อง จึงผ่านการตรวจยอดโดยไม่ลบข้อมูล

In [4]:
manifest_after = make_manifest(source_files)
if not manifest_before.equals(manifest_after):
    raise RuntimeError('ไฟล์ raw เปลี่ยนระหว่าง cleansing; ลบผลรอบนี้และรันใหม่จาก snapshot ที่คงที่')

audit_report = pd.DataFrame(audit_rows)
audit_path = output_dir / 'cleansing_audit.csv'
manifest_path = output_dir / 'source_manifest.csv'
audit_report.to_csv(audit_path, index=False)
manifest_after.to_csv(manifest_path, index=False)
display(audit_report)
print('รายงาน audit:', audit_path.relative_to(project_root))
print('manifest ต้นทาง:', manifest_path.relative_to(project_root))
print('จำนวนแถวที่ถูกลบ:', int(audit_report['rows_removed'].sum()))

,file,rows_in,rows_out,rows_removed,duplicate_rows_flagged,missing_cells,output_file
0,UNSW_NB15_testing-set.csv,82332,82332,0,0,0,UNSW_NB15_testing-set.csv
1,UNSW_NB15_training-set.csv,175341,175341,0,0,0,UNSW_NB15_training-set.csv


รายงาน audit: data-pipeline\UNSW-NB15\output\cleansing\cleansing_audit.csv
manifest ต้นทาง: data-pipeline\UNSW-NB15\output\cleansing\source_manifest.csv
จำนวนแถวที่ถูกลบ: 0


## 5. ตรวจให้เห็นการเปลี่ยนแปลงทีละ block และทีละไฟล์

บล็อกนี้แสดงหลายตารางต่อเนื่องกัน ไม่ใช่กราฟ: ตารางแรกเป็น audit ของแต่ละ chunk ใน Test; ตารางถัดไปเป็น chunk ของ Train (รอบนี้ Train ถูกแยกเป็น 100,000 และ 75,341 แถว); จากนั้นเป็นสรุปต่อไฟล์ และตารางท้ายจัดอันดับคอลัมน์ที่มีการเปลี่ยนเพื่อช่วยหาตำแหน่งที่ควรตรวจ ตารางทั้งหมดอ่าน raw เทียบกับ clean ตามตำแหน่งแถวเดิมเท่านั้น ไม่ได้ทำ cleansing ซ้ำ

วิธีอ่านคอลัมน์: `rows_in/rows_out/rows_removed` คือยอดเข้า-ออก-ลบ; `rows_with_changed_values` คือจำนวนแถวที่มีค่าเปลี่ยน; `changed_cells` คือจำนวนเซลล์ที่เปลี่ยน; `trimmed_cells` คือช่องที่ตัดช่องว่าง; `sentinel_to_missing_cells` คือ sentinel ที่แทนเป็น missing; `missing_cells_before/after` ใช้ยืนยันผลก่อน-หลัง; `duplicate_rows_flagged` คือแถวซ้ำที่ทำเครื่องหมายแต่ยังเก็บไว้; `orig_row_ids_added` คือจำนวนรหัสย้อนกลับที่เพิ่ม ตารางรายคอลัมน์ช่วยชี้ว่าคอลัมน์ไหนเปลี่ยนและเปลี่ยนแบบใด โดย metadata ที่เพิ่มไม่นำมานับเป็นการเปลี่ยนค่าต้นฉบับ

ผลรันนี้ทุก chunk และทั้งสองไฟล์มีแถวเข้าเท่ากับแถวออก: ลบ 0 แถว, changed/trim/sentinel/missing/duplicate เป็น 0 ทั้งหมด และเพิ่ม `orig_row_id` ครบทุก record; ตารางคอลัมน์จึงมีแต่ค่า changed เป็นศูนย์ ผลนี้บอกว่าไฟล์ official รุ่นปัจจุบันแทบไม่ต้องแก้ค่าเนื้อหา แต่ยังได้ไฟล์ clean ที่มี metadata และ audit สำหรับตรวจย้อนกลับ หากรันกับข้อมูลรุ่นใหม่ ตัวเลขที่ไม่เป็นศูนย์จะชี้ว่าควรตรวจ block และคอลัมน์ใดก่อนนำไปใช้

In [5]:
from itertools import zip_longest

def compare_as_cleaned(series):
    text = series.astype('string').str.strip()
    return text.mask(text.str.lower().isin(missing_words), pd.NA)

def clean_output_path(source_path):
    stem = source_path.name
    for suffix in ('.gz', '.csv', '.txt'):
        if stem.lower().endswith(suffix):
            stem = stem[:-len(suffix)]
    stem = re.sub(r'[^A-Za-z0-9_.-]+', '_', stem)
    return output_dir / f'{stem}.csv'

file_audit_details, column_change_details = [], []
for source_path in source_files:
    clean_path = clean_output_path(source_path)
    if not clean_path.is_file():
        raise FileNotFoundError(f'ไม่พบผล clean สำหรับเทียบ: {clean_path}')
    raw_reader = pd.read_csv(source_path, dtype='string', chunksize=chunk_size,
                             keep_default_na=False, na_filter=False)
    clean_reader = pd.read_csv(clean_path, dtype='string', chunksize=chunk_size,
                               keep_default_na=False, na_filter=False)
    totals = {key: 0 for key in [
        'rows_in', 'rows_out', 'rows_removed', 'changed_cells', 'rows_with_changed_values',
        'trimmed_cells', 'sentinel_to_missing_cells', 'missing_cells_before',
        'missing_cells_after', 'duplicate_rows_flagged', 'orig_row_ids_added',
    ]}
    per_column = {}
    chunk_number = 0
    for raw_part, clean_part in zip_longest(raw_reader, clean_reader):
        if raw_part is None or clean_part is None:
            raise ValueError(f'{source_path.name}: จำนวน chunk ของ raw/clean ไม่เท่ากัน')
        chunk_number += 1
        raw_part.columns = [str(name).replace('\ufeff', '').strip() for name in raw_part.columns]
        data_columns = [name for name in clean_part.columns if name not in {'orig_row_id', '_duplicate_flag'}]
        if list(raw_part.columns) != data_columns or len(raw_part) != len(clean_part):
            raise ValueError(f'{source_path.name}, chunk {chunk_number}: schema หรือจำนวนแถวไม่ตรงกัน')
        changed_mask = pd.DataFrame(False, index=raw_part.index, columns=raw_part.columns)
        chunk_trimmed = chunk_sentinel = chunk_missing_before = chunk_missing_after = 0
        for column in raw_part.columns:
            raw_text = raw_part[column].astype('string')
            trimmed = raw_text.str.strip()
            sentinel_mask = trimmed.str.lower().isin(missing_words)
            expected = trimmed.mask(sentinel_mask, pd.NA)
            actual = compare_as_cleaned(clean_part[column])
            changed = expected.fillna('<MISSING>').ne(actual.fillna('<MISSING>'))
            changed_mask[column] = changed.to_numpy()
            trimmed_count = int(raw_text.ne(trimmed).sum())
            sentinel_count = int((sentinel_mask & trimmed.ne('')).sum())
            before_missing = int(expected.isna().sum())
            after_missing = int(actual.isna().sum())
            chunk_trimmed += trimmed_count
            chunk_sentinel += sentinel_count
            chunk_missing_before += before_missing
            chunk_missing_after += after_missing
            column_item = per_column.setdefault(column, {
                'file': source_path.name, 'column': column, 'changed_cells': 0,
                'trimmed_cells': 0, 'sentinel_to_missing_cells': 0,
            })
            column_item['changed_cells'] += int(changed.sum())
            column_item['trimmed_cells'] += trimmed_count
            column_item['sentinel_to_missing_cells'] += sentinel_count
        rows_changed = int(changed_mask.any(axis=1).sum())
        duplicate_count = int(clean_part['_duplicate_flag'].str.lower().isin(['true', '1']).sum())
        chunk_rows = len(raw_part)
        chunk_audit = {
            'file': source_path.name, 'chunk': chunk_number, 'rows_in': chunk_rows,
            'rows_out': len(clean_part), 'rows_removed': chunk_rows - len(clean_part),
            'rows_with_changed_values': rows_changed,
            'changed_cells': int(changed_mask.to_numpy().sum()),
            'trimmed_cells': chunk_trimmed,
            'sentinel_to_missing_cells': chunk_sentinel,
            'missing_cells_before': chunk_missing_before,
            'missing_cells_after': chunk_missing_after,
            'duplicate_rows_flagged': duplicate_count,
            'orig_row_ids_added': int(clean_part['orig_row_id'].notna().sum()),
        }
        display(pd.DataFrame([chunk_audit]))
        for key in totals:
            totals[key] += chunk_audit[key]
    file_audit_details.append({'file': source_path.name, **totals,
                               'data_columns_added': 'orig_row_id, _duplicate_flag'})
    column_change_details.extend(per_column.values())

file_audit_details = pd.DataFrame(file_audit_details)
column_change_details = pd.DataFrame(column_change_details)
print('สรุปการเปลี่ยนแปลงแยกตามไฟล์:')
display(file_audit_details)
print('คอลัมน์ที่มีการเปลี่ยนแปลงมากที่สุด (ไม่นับแถวที่เพียงเพิ่ม metadata):')
display(column_change_details.sort_values(['changed_cells', 'trimmed_cells'], ascending=False).head(30))

# เพิ่มรายละเอียดลง audit เดิมเพื่อคงไฟล์สรุปชุดเดียว โดยไม่เปลี่ยน raw หรือจำนวนแถว
existing_audit = pd.read_csv(audit_path)
audit_report = existing_audit.drop(columns=[
    column for column in file_audit_details.columns if column != 'file' and column in existing_audit.columns
]).merge(file_audit_details, on='file', how='left')
audit_report.to_csv(audit_path, index=False)
print('อัปเดต cleansing audit:', audit_path.relative_to(project_root))
print('ข้อสรุป: duplicate ถูกทำเครื่องหมาย ไม่ได้ลบ; แถวที่ถูกลบจริง =', int(file_audit_details['rows_removed'].sum()))

,file,chunk,rows_in,rows_out,rows_removed,rows_with_changed_values,changed_cells,trimmed_cells,sentinel_to_missing_cells,missing_cells_before,missing_cells_after,duplicate_rows_flagged,orig_row_ids_added
0,UNSW_NB15_testing-set.csv,1,82332,82332,0,0,0,0,0,0,0,0,82332


,file,chunk,rows_in,rows_out,rows_removed,rows_with_changed_values,changed_cells,trimmed_cells,sentinel_to_missing_cells,missing_cells_before,missing_cells_after,duplicate_rows_flagged,orig_row_ids_added
0,UNSW_NB15_training-set.csv,1,100000,100000,0,0,0,0,0,0,0,0,100000


,file,chunk,rows_in,rows_out,rows_removed,rows_with_changed_values,changed_cells,trimmed_cells,sentinel_to_missing_cells,missing_cells_before,missing_cells_after,duplicate_rows_flagged,orig_row_ids_added
0,UNSW_NB15_training-set.csv,2,75341,75341,0,0,0,0,0,0,0,0,75341


สรุปการเปลี่ยนแปลงแยกตามไฟล์:


,file,rows_in,rows_out,rows_removed,changed_cells,rows_with_changed_values,trimmed_cells,sentinel_to_missing_cells,missing_cells_before,missing_cells_after,duplicate_rows_flagged,orig_row_ids_added,data_columns_added
0,UNSW_NB15_testing-set.csv,82332,82332,0,0,0,0,0,0,0,0,82332,"orig_row_id, _duplicate_flag"
1,UNSW_NB15_training-set.csv,175341,175341,0,0,0,0,0,0,0,0,175341,"orig_row_id, _duplicate_flag"


คอลัมน์ที่มีการเปลี่ยนแปลงมากที่สุด (ไม่นับแถวที่เพียงเพิ่ม metadata):


,file,column,changed_cells,trimmed_cells,sentinel_to_missing_cells
0,UNSW_NB15_testing-set.csv,id,0,0,0
1,UNSW_NB15_testing-set.csv,dur,0,0,0
2,UNSW_NB15_testing-set.csv,proto,0,0,0
3,UNSW_NB15_testing-set.csv,service,0,0,0
4,UNSW_NB15_testing-set.csv,state,0,0,0
5,UNSW_NB15_testing-set.csv,spkts,0,0,0
6,UNSW_NB15_testing-set.csv,dpkts,0,0,0
7,UNSW_NB15_testing-set.csv,sbytes,0,0,0
8,UNSW_NB15_testing-set.csv,dbytes,0,0,0
9,UNSW_NB15_testing-set.csv,rate,0,0,0


อัปเดต cleansing audit: data-pipeline\UNSW-NB15\output\cleansing\cleansing_audit.csv
ข้อสรุป: duplicate ถูกทำเครื่องหมาย ไม่ได้ลบ; แถวที่ถูกลบจริง = 0
